In [93]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [94]:
spark = SparkSession.builder \
    .appName("Lab2_Final_Complete") \
    .config("spark.jars", "/home/jovyan/jars/postgresql-42.7.2.jar,/home/jovyan/jars/clickhouse-jdbc-0.6.0-all.jar") \
    .config("spark.driver.extraClassPath", "/home/jovyan/jars/postgresql-42.7.2.jar:/home/jovyan/jars/clickhouse-jdbc-0.6.0-all.jar") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

pgUrl = "jdbc:postgresql://postgres:5432/postgres"
pgProperties = {"user": "postgres", "password": "postgres", "driver": "org.postgresql.Driver"}

chUrl = "jdbc:clickhouse://clickhouse:8123/default?use_binary_format=false&compress=false"
chProperties = {
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
    "user": "clickhouse",
    "password": "clickhouse",
    "create_table_engine": "Log"
}

In [95]:
def writeToClickHouse(df, tableName):
    df.write \
        .mode("overwrite") \
        .format("jdbc") \
        .option("url", chUrl) \
        .option("dbtable", tableName) \
        .option("user", chProperties["user"]) \
        .option("password", chProperties["password"]) \
        .option("driver", chProperties["driver"]) \
        .option("create_table_engine", "Log") \
        .save()
    print(f"Table {tableName} was written")

In [96]:
products = spark.read.jdbc(url=pgUrl, table="DimProducts", properties=pgProperties)
categories = spark.read.jdbc(url=pgUrl, table="DimProductCategories", properties=pgProperties)
customers = spark.read.jdbc(url=pgUrl, table="DimCustomers", properties=pgProperties)
geography = spark.read.jdbc(url=pgUrl, table="DimGeography", properties=pgProperties)
stores = spark.read.jdbc(url=pgUrl, table="DimStores", properties=pgProperties)
suppliers = spark.read.jdbc(url=pgUrl, table="DimSuppliers", properties=pgProperties)
sales = spark.read.jdbc(url=pgUrl, table="FactSales", properties=pgProperties)

In [97]:
productBase = sales.join(products, "productId") \
    .join(categories, "categoryId")

# Средний рейтинг и количество отзывов для каждого продукта
productSales = productBase.groupBy("productId", products["name"], categories["categoryName"]) \
    .agg(
        F.avg("rating").alias("avgRating"),
        F.sum("reviews").alias("totalReviews")
    )
# Топ-10 самых продаваемых продуктов
top10Products = productBase.groupBy(products["name"]) \
    .agg(F.sum("quantity").alias("totalQuantity")) \
    .orderBy(F.desc("totalQuantity")).limit(10)
# Общая выручка по категориям продуктов
revenueByCategory = productBase.groupBy(categories["categoryName"]) \
    .agg(F.sum("totalPrice").alias("categoryRevenue")) \
    .orderBy(F.desc("categoryRevenue"))

writeToClickHouse(top10Products, "Top10Products")
writeToClickHouse(revenueByCategory, "RevenueByCategory")
writeToClickHouse(productSales, "ReportProductSales")

Table Top10Products was written
Table RevenueByCategory was written
Table ReportProductSales was written


In [98]:
customerBase = sales.join(customers, "customerId") \
    .join(geography, "geoId")

# Средний чек для каждого клиента
customerSales = customerBase.groupBy("customerId", "firstName", "lastName", "country") \
    .agg(F.avg("totalPrice").alias("avgCheck"))
# Топ-10 клиентов с наибольшей общей суммой покупок
top10Customers = customerBase.groupBy("customerId", "firstName", "lastName") \
    .agg(F.sum("totalPrice").alias("totalSpent")) \
    .orderBy(F.desc("totalSpent")).limit(10)
# Распределение клиентов по странам
customerDistribution = customerBase.groupBy("country") \
    .agg(F.countDistinct("customerId").alias("customerCount"),
         F.sum("totalPrice").alias("totalRevenueByCountry")) \
    .orderBy(F.desc("customerCount"))

writeToClickHouse(top10Customers, "Top10Customers")
writeToClickHouse(customerDistribution, "CustomerDistributionByCountry")
writeToClickHouse(customerSales, "ReportCustomerSales")

Table Top10Customers was written
Table CustomerDistributionByCountry was written
Table ReportCustomerSales was written


In [99]:
# Средний размер заказа по месяцам
timeSales = sales.withColumn("year", F.year("saleDate")) \
    .withColumn("month", F.month("saleDate")) \
    .groupBy("year", "month") \
    .agg (
        F.sum("totalPrice").alias("monthlyRevenue"),
        F.avg("totalPrice").alias("avgOrderSize")
    ) \
    .orderBy("year", "month")
# Месячные и годовые тренды продаж
# Сравнение выручки за разные периоды
windowSpec = Window.orderBy("year", "month")
timeSalesWithLag = timeSales.withColumn("prevMonthRevenue", F.lag("monthlyRevenue").over(windowSpec)) \
    .withColumn("revenueChange", F.col("monthlyRevenue") - F.col("prevMonthRevenue")) \
    .withColumn("revenueChangePercent", 
                F.when(F.col("prevMonthRevenue").isNotNull(),
                       (F.col("monthlyRevenue") - F.col("prevMonthRevenue")) / F.col("prevMonthRevenue") * 100)
                .otherwise(F.lit(0)))
timeSalesWithLag = timeSalesWithLag.fillna(0, subset=["prevMonthRevenue", "revenueChange", "revenueChangePercent"])

writeToClickHouse(timeSalesWithLag, "ReportTimeSales")

Table ReportTimeSales was written


In [100]:
storeBase = sales.join(stores, "storeId").join(geography, "geoId")

# Средний чек для каждого магазина
storeSales = storeBase.groupBy(stores["name"], "city", "country") \
    .agg(F.sum("totalPrice").alias("storeRevenue"),
         F.avg("totalPrice").alias("storeAvgCheck"))
# Топ-5 магазинов с наибольшей выручкой
top5Stores = storeBase.groupBy(stores["name"]) \
    .agg(F.sum("totalPrice").alias("storeRevenue")) \
    .orderBy(F.desc("storeRevenue")).limit(5)
# Распределение продаж по городам и странам
salesByCityCountry = storeBase.groupBy("country", "city") \
    .agg(F.sum("totalPrice").alias("totalRevenue"),
         F.avg("totalPrice").alias("avgCheck")) \
    .orderBy("country", F.desc("totalRevenue"))

writeToClickHouse(top5Stores, "Top5Stores")
writeToClickHouse(salesByCityCountry, "SalesByCityCountry")
writeToClickHouse(storeSales, "ReportStoreSales")

Table Top5Stores was written
Table SalesByCityCountry was written
Table ReportStoreSales was written


In [101]:
supplierBase = sales.join(suppliers, "supplierId") \
    .join(geography, "geoId") \
    .join(products, "productId")

supplierRevenue = supplierBase.groupBy(suppliers["name"], "country") \
    .agg(F.sum("totalPrice").alias("supplierRevenue"))

# Средняя цена товаров от каждого поставщика
supplierAvgPrice = supplierBase.groupBy(suppliers["name"]) \
    .agg(F.avg(products["price"]).alias("avgProductPrice"))
# Топ-5 поставщиков с наибольшей выручкой
top5Suppliers = supplierRevenue.orderBy(F.desc("supplierRevenue")).limit(5)
# Распределение продаж по странам поставщиков
supplierDistribution = supplierRevenue.groupBy("country") \
    .agg(F.sum("supplierRevenue").alias("revenueByCountry"),
         F.countDistinct(suppliers["name"]).alias("supplierCount"))

writeToClickHouse(top5Suppliers, "Top5Suppliers")
writeToClickHouse(supplierDistribution, "SupplierDistributionByCountry")
writeToClickHouse(supplierAvgPrice, "SupplierAvgPrice")

Table Top5Suppliers was written
Table SupplierDistributionByCountry was written
Table SupplierAvgPrice was written


In [102]:
productSalesVolume = sales.groupBy("productId").agg(F.sum("quantity").alias("salesVolume"))

qualityBase = products.join(productSalesVolume, "productId", "left") \
    .select(products["name"], "rating", "reviews", "salesVolume") \
    .fillna(0, subset=["salesVolume"])

# Продукты с наибольшим количеством отзывов
topReviewsProducts = qualityBase.orderBy(F.desc("reviews")).limit(10)
# Корреляция между рейтингом и объемом продаж
correlation = qualityBase.agg(F.corr("rating", "salesVolume").alias("rating_sales_correlation"))
# Продукты с наивысшим и наименьшим рейтингомin)
maxRatingProduct = qualityBase.orderBy(F.desc("rating")).limit(1)
minRatingProduct = qualityBase.orderBy(F.asc("rating")).limit(1)

writeToClickHouse(correlation, "RatingSalesCorrelation")
writeToClickHouse(maxRatingProduct, "MaxRatingProduct")
writeToClickHouse(minRatingProduct, "MinRatingProduct")
writeToClickHouse(topReviewsProducts, "TopReviewsProducts")

Table RatingSalesCorrelation was written
Table MaxRatingProduct was written
Table MinRatingProduct was written
Table TopReviewsProducts was written
